# PTEN abundance scores (MaveDB `urn:mavedb:00000013-a-1`)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

scores = pd.read_csv("data/derived/NILAYBAD_mavedb-00000013-a-1-scores.csv")
lines = open("data/derived/pten-reference-protein.fasta").read().split()
ref_seq = "".join(l for l in lines if not l.startswith(">"))

## Explain what each row and column means in the score tables

In [39]:
scores.head()

,accession,hgvs_nt,hgvs_splice,hgvs_pro,score,sd,expts,se,lower_ci,upper_ci,...,exp1_w_ave,exp2_w_ave,exp3_w_ave,exp4_w_ave,exp5_w_ave,exp6_w_ave,exp7_w_ave,exp8_w_ave,snv,abundance_class
0,urn:mavedb:00000013-a-1#1,NaN,NaN,p.Met1val,1.065143,0.128434,4,0.064217,0.939281,1.191006,...,0.849318,0.823041,NaN,NaN,0.802659,0.721626,NaN,NaN,0,4
1,urn:mavedb:00000013-a-1#2,NaN,NaN,p.Thr2Thr,0.824359,0.428489,7,0.161954,0.506935,1.141782,...,0.436220,0.875207,0.838369,0.834303,0.525840,0.518268,NaN,0.836967,1,3
2,urn:mavedb:00000013-a-1#3,NaN,NaN,p.Asn12Ile,0.298717,0.272272,7,0.102909,0.097019,0.500415,...,0.538292,0.396138,0.533358,0.592458,0.369006,NaN,0.268813,0.452092,1,1
3,urn:mavedb:00000013-a-1#4,NaN,NaN,p.R74Val,0.855027,0.318442,6,0.130003,0.600225,1.109829,...,0.630357,0.824519,0.827260,0.732211,0.487763,NaN,NaN,0.776376,0,3
4,urn:mavedb:00000013-a-1#5,NaN,NaN,p.His75Ala,0.863776,0.223173,8,0.078903,0.709128,1.018423,...,0.788377,0.559483,0.711924,0.537434,0.687006,0.872204,0.828631,0.769184,0,4


| column | meaning |
|---|---|
| `accession` | MaveDB ID for the row |
| `hgvs_nt`, `hgvs_splice` | nucleotide / splice labels (empty in this dataset) |
| `hgvs_pro` | protein change, e.g. `p.Met1Val` = position 1, Met changed to Val |
| `score` | abundance score averaged over replicates; WT = 1, fully degraded ≈ 0 |
| `sd` | standard deviation of the score |
| `expts` | number of replicate experiments that measured the variant |
| `se` | standard error of the score |
| `lower_ci`, `upper_ci` | 95% confidence interval of the score |
| `score1`-`score8` | score in each of the 8 replicates |
| `median_w_ave`, `exp1_w_ave`-`exp8_w_ave` | raw weighted-average fluorescence bin (0.25-1) that the score is computed from |
| `snv` | 1 if reachable by a single nucleotide change |
| `abundance_class` | 1 = low, 2 = possibly low, 3 = possibly WT-like, 4 = WT-like |

## Parse mutation labels in the score tables to identify the site, WT residue, and mutant, and verify that the listed WT residue matches the reference

Split each `hgvs_pro` label into WT residue, position and mutant (`Ter` = stop, `=` = no change).

In [44]:
parts = scores["hgvs_pro"].str.extract(r"p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|=)")
scores["wt"], scores["pos"], scores["mut"] = parts[0], parts[1].astype(float), parts[2]
scores[["hgvs_pro", "wt", "pos", "mut"]].head(15)

,hgvs_pro,wt,pos,mut
0,p.Met1val,NaN,NaN,NaN
1,p.Thr2Thr,Thr,2.0,Thr
2,p.Asn12Ile,Asn,12.0,Ile
3,p.R74Val,NaN,NaN,NaN
4,p.His75Ala,His,75.0,Ala
5,p.His75Asp,His,75.0,Asp
6,His75Glu,NaN,NaN,NaN
7,p.His75Phe,His,75.0,Phe
8,p.His75Gly,His,75.0,Gly
9,p.His75Ile,His,75.0,Ile


Label each row as WT, synonymous, nonsense or missense based on the mutant.

In [41]:
def classify(row):
    if row["hgvs_pro"] == "_wt": return "WT"
    if row["mut"] == "=" or row["mut"] == row["wt"]: return "synonymous"
    if row["mut"] == "Ter": return "nonsense"
    return "missense"

scores["type"] = scores.apply(classify, axis=1)
scores["type"].value_counts()

,count
type,
missense,4132
synonymous,142
nonsense,134
WT,1


Convert the 3-letter WT residue to 1 letter and compare it to the reference sequence at that position.

In [42]:
letter = {"Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E","Gly":"G","His":"H","Ile":"I",
      "Leu":"L","Lys":"K","Met":"M","Phe":"F","Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V"}

variants = scores[scores["type"] != "WT"].copy()
variants["ref"] = variants["pos"].astype(int).map(lambda p: ref_seq[p - 1])
variants["match"] = variants["wt"].map(letter) == variants["ref"]
print(variants["match"].value_counts())
print("total:", variants["match"].value_counts().sum())


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

## 3. Compare score distributions for WT, missense, nonsense, and synonymous mutations

In [ ]:
scores.groupby("type")["score"].describe()

Box plot of the scores for each variant type.

In [ ]:
order = ["WT", "synonymous", "missense", "nonsense"]
plt.boxplot([scores.loc[scores["type"] == t, "score"] for t in order], tick_labels=order)
plt.title("PTEN abundance score by variant type")
plt.ylabel("score")
plt.show()